# ML-04 — Search Intelligence Data Contract

**Lane:** Content refresh / declining-page prioritization

This notebook defines the warehouse grain, verifies three facts on February 2026, builds five pre-decision features, and demonstrates one deliberate label-leakage trap.


## 1. Unit of analysis + time window

**1) What one row means:** One row in `fact_content_daily_performance` is one daily observation for one pseudonymized content item and client: `report_date × client_hash_id × content_hash_id`.

**2) Tables:** I use `fact_content_daily_performance` for daily performance. `dim_content` is context only if a content attribute is needed.

**3) Time window:** February 2026 is the feature/decision window. March 2026 is the future outcome window. Decision cutoff: **2026-02-28**.

**4) What I predict:** Whether a page goes dark in March 2026, defined as zero measured GSC clicks during March (`went_dark`).

**5) Deliberately excluded:** March/future information, including the March label, because it is not knowable at the February decision moment.


In [ ]:
%pip -q install duckdb pandas scikit-learn

import os, getpass, duckdb, numpy as np, pandas as pd

def get_hf_token():
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Hugging Face READ token (not stored in notebook): " )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [get_hf_token()])
con.execute("""
CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
print("Connected. Feature window: February 2026 | Label window: March 2026")


## 2. Fields: feature / label / context / excluded

**Five features**
1. `feb_impressions` — February GSC impressions. **Available when?** By 2026-02-28.
2. `feb_clicks` — February GSC clicks. **Available when?** By the cutoff.
3. `feb_ctr` — February clicks / impressions. **Available when?** Computed only from February observations.
4. `feb_avg_position` — impression-weighted GSC position in February. **Available when?** From February observations only.
5. `feb_measured_days` — February days with `gsc_data_available IS TRUE`. **Available when?** By the cutoff.

**Label / proxy:** `went_dark` — 1 when measured March GSC clicks equal zero; 0 otherwise. Future information, never a feature.

**Context:** `client_hash_id`, `content_hash_id`, `report_date` are keys/time fields for grouping and joins, not model features.

**Excluded:** all March/future fields from feature construction, product-generated decision flags, and the final June `_sample` table.


## 3. Verify the contract with exactly three checks

### Query 1 — Grain
If the claimed grain is correct, there should be no duplicate `client × content × date` keys.


In [ ]:
grain_check = con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
FROM {FEB}
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print(f"Duplicate grain keys returned: {len(grain_check)}")
grain_check


### Query 2 — February row count and date span

In [ ]:
window_check = con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM {FEB}
""").df()
window_check


### Query 3 — GSC availability
The availability check deliberately uses `IS TRUE`. FALSE and NULL are not counted as measured GSC observations.


In [ ]:
availability_check = con.sql(f"""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS measured_gsc_rows,
  ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / NULLIF(COUNT(*),0), 2) AS measured_pct
FROM {FEB}
""").df()
availability_check


## Five-feature frame
The frame is aggregated to the content-item × client decision grain. Every feature is built only from February observations, so it is available at the 2026-02-28 cutoff.


In [ ]:
features = con.sql(f"""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
  SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
  100.0 * SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE),0) AS feb_ctr,
  SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE),0) AS feb_avg_position,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS feb_measured_days
FROM {FEB}
GROUP BY 1,2
""").df()
feature_cols = ["feb_impressions","feb_clicks","feb_ctr","feb_avg_position","feb_measured_days"]
print("Feature frame shape:", features.shape)
print("Five features:", feature_cols)
features.head()


## 4. The trap — deliberately leak the label
Create the March outcome from future data, then intentionally add that outcome as a feature. The leaked score is invalid; the honest score is the one kept.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

labels = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS mar_clicks,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS mar_measured_days
FROM {MAR}
GROUP BY 1,2
HAVING COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) > 0
""").df()
labels["went_dark"] = (labels["mar_clicks"] == 0).astype(int)
frame = features.merge(labels[["client_hash_id","content_hash_id","went_dark"]], on=["client_hash_id","content_hash_id"], how="inner")

X, y = frame[feature_cols], frame["went_dark"]
Xtr, Xte, ytr, yte = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
model = make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LogisticRegression(max_iter=1000,random_state=42))
model.fit(Xtr,ytr)
honest_accuracy = accuracy_score(yte, model.predict(Xte))

# Deliberate leakage: the target itself is included as a feature.
X_leaky = frame[feature_cols + ["went_dark"]]
Xtr, Xte, ytr, yte = train_test_split(X_leaky,y,test_size=.25,random_state=42,stratify=y)
leaky_model = make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LogisticRegression(max_iter=1000,random_state=42))
leaky_model.fit(Xtr,ytr)
leaky_accuracy = accuracy_score(yte, leaky_model.predict(Xte))

print(f"Honest accuracy (five pre-cutoff features): {honest_accuracy:.3f}")
print(f"Leaky accuracy (target included as a feature): {leaky_accuracy:.3f}")
print(f"Leakage jump: {leaky_accuracy-honest_accuracy:+.3f}")


### Leakage result and cleanup
`went_dark` is the answer we are trying to predict, so including it as a feature makes the score artificially strong. **Keep the honest score; remove `went_dark` from the feature matrix.** This is the same failure mode as using `trend_direction` / `trend_pct` in the starter-data exercise.


## 5. Limitation

**Named limitation — uneven historical coverage.** GSC availability is not uniform across clients and dates. Missing/unavailable GSC observations are not equivalent to zero impressions or zero clicks, so the analysis uses `gsc_data_available IS TRUE`. The resulting lane therefore describes pages with measured GSC history, not every page in the warehouse.

The February → March `went_dark` definition is also a specific one-month outcome; it should not automatically be treated as a universal definition of content quality or decline.

## Self-check

- [x] Five contract answers are stated.
- [x] Exactly three verification checks are shown; availability uses `IS TRUE`.
- [x] Five features maximum, each with an “available when?” line.
- [x] One deliberate label-leak experiment is shown and the label is removed from the honest feature set.
- [x] One named limitation is documented.
- [ ] Run top-to-bottom in Colab after setting `HF_TOKEN`, then commit the **executed** notebook.
- [ ] Confirm no token or client-identifying data appears in outputs.
